# Task: GNN training on synthetic SPV simulations: adjacency, cell state and property matrices

We will be developing a graph neural network (GNN)-based model capable of inferring mechanistic rules and uncovering the principles driving DPAC aggregation. To facilitate this, the GNN will initially be trained using synthetic Self-Propelled Voronoi (SPV) simulations, serving as placeholder data while the deep learning infrastructure is optimized. The GNN will be validated by its ability to, first, recover the physical mechanisms embedded in the SPV model, then subsequently applied to DPAC data to explore the impacts of initial thickness and cell density.

### GNN training

In [38]:
import os
import numpy as np
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GINEConv
from torch_geometric.utils import from_networkx

from torch.optim import AdamW, lr_scheduler
from sklearn.model_selection import KFold

##########################################################################
# Custom collate function for standard DataLoader
##########################################################################

def pyg_collate_fn(batch_list):
    """
    Merges a list of PyG Data objects into a single `Batch`.
    Useful when using the standard torch.utils.data.DataLoader.
    """
    return Batch.from_data_list(batch_list)

##########################################################################
# 1. Parameter Parsing
##########################################################################

def parse_parameters(param_path):
    """
    Parse parameters from the given file.
    Ensures that W is converted to a NumPy array if it isn't already.
    """
    params = {}
    print(f"Parsing parameters from {param_path}")
    with open(param_path, "r") as f:
        exec(f.read(), {}, params)  # Execute the file content in a controlled namespace
    if not isinstance(params["W"], np.ndarray):
        params["W"] = np.array(params["W"])  # Convert W to NumPy array
    print(f"Successfully parsed parameters: {params}")
    return params

##########################################################################
# 2. Data Loading with Caching
##########################################################################

def load_matrices(directory, timepoint):
    """Load matrices with caching for faster subsequent access"""
    cache_key = (directory, timepoint)
    if not hasattr(load_matrices, 'cache'):
        load_matrices.cache = {}
    if cache_key not in load_matrices.cache:
        print(f"Loading matrices for timepoint {timepoint} from {directory}")
        graph_mat = np.load(os.path.join(directory, f"{timepoint}_graph_mat.npy"))
        properties_mat = np.load(os.path.join(directory, f"{timepoint}_properties_mat.npy"))
        state_mat = np.load(os.path.join(directory, f"{timepoint}_state_mat.npy"))
        load_matrices.cache[cache_key] = (graph_mat, properties_mat, state_mat)
    return load_matrices.cache[cache_key]

##########################################################################
# 3. GCA Initialization
##########################################################################

def initialize_gca(graph_mat, properties_mat, state_mat, params):
    """
    Create a graph (GCA) with proper state handling and parameter validation.
    """
    print("Initializing GCA graph...")
    g = nx.from_numpy_array(graph_mat, create_using=nx.Graph)
    
    # Validate physical constraints
    if np.any(properties_mat[:, 0] < 0) or np.any(properties_mat[:, 1] < 0):
        raise ValueError("Area and perimeter must be non-negative")

    # Add node properties
    for i, (area, perimeter) in enumerate(properties_mat):
        if area < 0 or perimeter < 0:
            raise ValueError(f"Invalid properties at node {i}: area={area}, perimeter={perimeter}")
        
        cell_type = np.argmax(state_mat[i])
        g.nodes[i]["state"] = state_mat[i]   # Full state vector
        g.nodes[i].update({
            "area": max(area, 0),
            "perimeter": max(perimeter, 0),
            "motility": params["v0"][cell_type],  # from v0 array
            "persistence": params["Dr"],
            "kappa_A": params["kappa_A"],
            "kappa_P": params["kappa_P"],
            "A0": params["A0"][cell_type],
            "P0": params["P0"][cell_type],
        })

    # Add edge properties
    for u, v in g.edges():
        type_u = np.argmax(g.nodes[u]["state"])
        type_v = np.argmax(g.nodes[v]["state"])
        adhesion = params["W"][type_u][type_v]
        g.edges[u, v].update({
            "adhesion": adhesion,
            "repulsion_radius": params["a"], 
            "repulsion_coefficient": params["k"],
        })

    g.graph["adj_matrix"] = graph_mat
    return g

##########################################################################
# 4. Define the GraphPredictor Model (with the FIX)
##########################################################################

class GraphPredictor(nn.Module):
    """
    A GNN using GINEConv to handle multi-dimensional edge_attr.
    Predicts:
      1) Next cell-state distribution (state probabilities),
      2) Next area and perimeter,
      3) Changes in adjacency.
    """
    def __init__(self, node_dim, edge_dim, hidden_dim, num_cell_types):
        super().__init__()

        # 1) Node encoder: transforms raw node_dim -> hidden_dim
        self.node_encoder = nn.Linear(node_dim, hidden_dim)

        # 2) MLP for node embeddings (used by GINEConv for neighbor aggregation)
        #    This MLP expects input dimension = hidden_dim -> output = hidden_dim.
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # 3) Two GINEConv layers
        #    Passing 'edge_dim=...' tells GINEConv to internally build a Linear(3->hidden_dim) for edges.
        self.conv1 = GINEConv(nn=self.node_mlp, edge_dim=edge_dim)
        self.conv2 = GINEConv(nn=self.node_mlp, edge_dim=edge_dim)
        
        # 4) Output decoders
        self.state_decoder = nn.Linear(hidden_dim, num_cell_types)
        self.area_decoder  = nn.Linear(hidden_dim, 1)
        self.perim_decoder = nn.Linear(hidden_dim, 1)

        # For adjacency changes, we compute a pairwise embedding:
        self.adj_decoder   = nn.Linear(2 * hidden_dim, 1)

    def forward(self, x, edge_index, edge_attr, batch=None):
        """
        Forward pass. Returns (state_pred, area_pred, perim_pred, adj_pred).
        """
        # Encode node features from node_dim -> hidden_dim
        x = self.node_encoder(x)   # [num_nodes, hidden_dim]
        
        # GINEConv #1
        x = self.conv1(x, edge_index, edge_attr)  
        x = F.relu(x)

        # GINEConv #2
        x = self.conv2(x, edge_index, edge_attr)
        x = F.relu(x)
        
        # Decode per-node predictions
        state_pred = F.log_softmax(self.state_decoder(x), dim=-1)  # [num_nodes, num_cell_types]
        area_pred  = self.area_decoder(x)                          # [num_nodes, 1]
        perim_pred = self.perim_decoder(x)                         # [num_nodes, 1]
        
        # For adjacency changes, build edge representations
        row, col = edge_index
        x_u = x[row]   # [num_edges, hidden_dim]
        x_v = x[col]   # [num_edges, hidden_dim]
        edge_repr = torch.cat([x_u, x_v], dim=1)        # [num_edges, 2*hidden_dim]
        adj_pred  = self.adj_decoder(edge_repr)         # [num_edges, 1]
        
        return state_pred, area_pred, perim_pred, adj_pred

##########################################################################
# 5. Training & Validation
##########################################################################

def train_epoch(model, data_list, optimizer, device, num_cell_types):
    model.train()
    total_loss = 0
    
    loader = DataLoader(data_list, batch_size=1, shuffle=True, collate_fn=pyg_collate_fn)
    
    criterion_state = nn.KLDivLoss(reduction='batchmean')
    criterion_prop  = nn.SmoothL1Loss()
    criterion_adj   = nn.BCEWithLogitsLoss()
    
    for batch_idx, batch in enumerate(loader):
        optimizer.zero_grad()
        
        x = batch.x.to(device)
        edge_index = batch.edge_index.to(device)
        edge_attr = batch.edge_attr.to(device)
        
        next_state = batch.next_state.to(device)
        next_area  = batch.next_area.unsqueeze(-1).to(device)
        next_perim = batch.next_perim.unsqueeze(-1).to(device)
        next_adj   = batch.next_adj.to(device)
        
        state_pred, area_pred, perim_pred, adj_pred = model(x, edge_index, edge_attr)
        
        loss_state = criterion_state(state_pred, next_state)
        loss_area  = criterion_prop(area_pred, next_area)
        loss_perim = criterion_prop(perim_pred, next_perim)
        loss_adj   = criterion_adj(adj_pred.squeeze(), next_adj.float())
        
        total_batch_loss = loss_state + loss_area + loss_perim + loss_adj

        # Optional L2 regularization
        l2_reg = 1e-4 * sum(p.pow(2.0).sum() for p in model.parameters())
        total_batch_loss += l2_reg
        
        total_batch_loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += total_batch_loss.item()
        
        if (batch_idx + 1) % 10 == 0:
            print(f"[Train] Batch {batch_idx + 1}/{len(loader)} - Loss: {total_batch_loss.item():.4f}")
    
    avg_loss = total_loss / len(loader)
    print(f"[Train] Epoch complete. Average loss: {avg_loss:.4f}")
    return avg_loss

def validate(model, data_list, device, num_cell_types):
    model.eval()
    total_loss = 0
    
    loader = DataLoader(data_list, batch_size=1, shuffle=False, collate_fn=pyg_collate_fn)
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            x = batch.x.to(device)
            edge_index = batch.edge_index.to(device)
            edge_attr = batch.edge_attr.to(device)
            
            next_state = batch.next_state.to(device)
            next_area  = batch.next_area.unsqueeze(-1).to(device)
            next_perim = batch.next_perim.unsqueeze(-1).to(device)
            next_adj   = batch.next_adj.to(device)
            
            state_pred, area_pred, perim_pred, adj_pred = model(x, edge_index, edge_attr)
            
            loss_state = F.kl_div(state_pred, next_state, reduction='batchmean')
            loss_area  = F.l1_loss(area_pred, next_area)
            loss_perim = F.l1_loss(perim_pred, next_perim)
            loss_adj   = F.binary_cross_entropy_with_logits(adj_pred.squeeze(), next_adj.float())
            
            total_loss += (loss_state + loss_area + loss_perim + loss_adj).item()
            
            if (batch_idx + 1) % 10 == 0:
                print(f"[Val] Batch {batch_idx + 1}/{len(loader)} - Cumulative Loss: {total_loss:.4f}")
    
    avg_loss = total_loss / len(loader)
    print(f"[Val] Validation complete. Average loss: {avg_loss:.4f}")
    return avg_loss

##########################################################################
# 6. K-Fold Training
##########################################################################

def k_fold_training(
    data_dirs, param_files, timepoints, timepoint_interval,
    node_dim, edge_dim, hidden_dim, num_cell_types,
    epochs=100, lr=1e-3, k_folds=5
):
    """
    data_dirs, param_files: lists of directories and param files for each dataset
    timepoints: total number of timepoints you have
    timepoint_interval: spacing between timepoints for constructing (current -> next) pairs
    node_dim, edge_dim, hidden_dim: dimensionalities for the GNN
    num_cell_types: how many cell types are possible
    epochs, lr, k_folds: training hyperparameters
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Preprocess all data
    print("Preprocessing data...")
    all_data = []
    for dir_idx, (data_dir, param_file) in enumerate(zip(data_dirs, param_files)):
        print(f"Processing directory {data_dir} with parameter file {param_file}...")
        params = parse_parameters(param_file)
        
        # Build training pairs for each consecutive interval
        #
        # NOTE: If you want the last pair (e.g. (1500,2000) for 2000 total timepoints),
        #       consider using: for t in range(0, timepoints, timepoint_interval): 
        #       and handle the boundary if needed.
        for t in range(0, timepoints - timepoint_interval, timepoint_interval):
            print(f"Loading timepoint {t} and {t + timepoint_interval}...")
            current = load_matrices(data_dir, t)
            next_t = load_matrices(data_dir, t + timepoint_interval)
            
            # Initialize GCAs
            g_current = initialize_gca(*current, params)
            g_next = initialize_gca(*next_t, params)
            
            # Convert to PyG data
            data = from_networkx(g_current)

            # Add node features: area, perimeter, state
            node_feat = []
            for i in g_current.nodes:
                node_feat.append(np.concatenate([
                    [g_current.nodes[i]["area"], g_current.nodes[i]["perimeter"]],
                    g_current.nodes[i]["state"]
                ]))
            data.x = torch.tensor(node_feat, dtype=torch.float)

            # Add edge features: adhesion, repulsion_radius, repulsion_coefficient
            edge_feat = []
            for (u, v) in zip(data.edge_index[0], data.edge_index[1]):
                edge = g_current.edges[int(u), int(v)]
                edge_feat.append([
                    edge["adhesion"], 
                    edge["repulsion_radius"], 
                    edge["repulsion_coefficient"]
                ])
            data.edge_attr = torch.tensor(edge_feat, dtype=torch.float)

            # Add target attributes (next state, area, perimeter, adjacency)
            data.next_state = torch.tensor(
                [g_next.nodes[i]["state"] for i in g_current.nodes],
                dtype=torch.float
            )
            data.next_area = torch.tensor(
                [g_next.nodes[i]["area"] for i in g_current.nodes],
                dtype=torch.float
            )
            data.next_perim = torch.tensor(
                [g_next.nodes[i]["perimeter"] for i in g_current.nodes],
                dtype=torch.float
            )
            data.next_adj = torch.tensor(
                nx.to_numpy_array(g_next)[data.edge_index[0], data.edge_index[1]],
                dtype=torch.float
            )

            # Quick debug checks for each data object:
            print(f"  Created PyG Data object with keys: {list(data.keys())}")
            print(f"  -> x.shape={data.x.shape}, edge_attr.shape={data.edge_attr.shape}")
            print(f"  -> next_state.shape={data.next_state.shape}, next_area.shape={data.next_area.shape}, "
                  f"next_perim.shape={data.next_perim.shape}, next_adj.shape={data.next_adj.shape}\n")

            # Ensure 'data' is a single PyG Data object
            if isinstance(data, Data):
                all_data.append(data)
            else:
                raise ValueError(f"Invalid data type: {type(data)}. Expected torch_geometric.data.Data.")

    print(f"Total number of samples in all_data: {len(all_data)}")
    
    # K-fold training
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    best_model = None
    best_loss = float('inf')
    
    data_array = list(all_data)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(data_array)):
        print(f"\n--- Starting Fold {fold+1}/{k_folds} ---")
        train_data = [data_array[i] for i in train_idx]
        val_data = [data_array[i] for i in val_idx]
        
        print(f"  Training data size: {len(train_data)}")
        print(f"  Validation data size: {len(val_data)}")
        
        # Initialize model, optimizer, scheduler
        model = GraphPredictor(node_dim, edge_dim, hidden_dim, num_cell_types).to(device)
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
        
        best_val_loss = float('inf')
        patience_counter = 0
        for epoch in range(epochs):
            print(f"\nEpoch {epoch+1}/{epochs}")
            train_loss = train_epoch(model, train_data, optimizer, device, num_cell_types)
            val_loss = validate(model, val_data, device, num_cell_types)
            scheduler.step(val_loss)
            
            print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            
            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                torch.save(model.state_dict(), f"best_fold{fold}.pth")
                print(f"    --> New best model saved for fold {fold+1} with validation loss: {best_val_loss:.4f}")
            else:
                patience_counter += 1
                if patience_counter >= 10:
                    print("    --> Early stopping triggered.")
                    break
        
        # Update best overall model
        if best_val_loss < best_loss:
            best_loss = best_val_loss
            best_model = model
    
    print(f"\nTraining complete. Best validation loss across folds: {best_loss:.4f}")
    
    if best_model is not None:
        torch.save(best_model.state_dict(), "best_model.pth")
        print("Best model saved to 'best_model.pth'")
    else:
        print("No best model found (should not happen unless no training occurred).")
    
    return best_model

##########################################################################
# 7. Main Execution
##########################################################################

if __name__ == "__main__":

    
    # Suppose 10 data directories, each with an associated param file:
    data_dirs = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/{i}_matrix_output"
        for i in range(1, 11)
    ]
    param_files = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/{i}.py"
        for i in range(1, 11)
    ]

    timepoints = 2000
    timepoint_interval = 500
    
    num_cell_types = 2
    node_dim = 2 + num_cell_types  # (area, perimeter) + cell-state distribution
    edge_dim = 3                   # (adhesion, repulsion_radius, repulsion_coefficient)
    hidden_dim = 64
    
    best_model = k_fold_training(
        data_dirs, param_files,
        timepoints, timepoint_interval,
        node_dim, edge_dim, hidden_dim, num_cell_types,
        epochs=30, lr=1e-3, k_folds=3
    )

    print("Done.")



Using device: cpu
Preprocessing data...
Processing directory /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/1_matrix_output with parameter file /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py...
Parsing parameters from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py
Successfully parsed parameters: {'domain_size': [60, 14], 'init_noise': 0.005, 'rng_seed': 1, 'dt': 0.25, 'tMax': 501, 'stripe_thickness': 4, 'stripe_density': 0.5, 'v0': [0.1, 1.3], 'W': array([[0.  , 0.08],
       [0.08, 0.  ]]), 'A0': [0.9, 0.9], 'P0': [3.812, 3.812], 'Dr': 50, 'kappa_A': 0.4, 'kappa_P': 0.07, 'a': 0.25, 'k': 2.5}
Loading timepoint 0 and 500...
Loading matrices for timepoint 0 from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/1_matrix_output
Loading matrices for timepoint 500 from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output

In [2]:
import torch
from torch.utils.data import DataLoader
from torch_geometric.data import Data, Batch

def pyg_collate_fn(batch_list):
    return Batch.from_data_list(batch_list)

# Make a single small Data object
data = Data(x=torch.rand(3,2), edge_index=torch.tensor([[0,1],[1,2]]))

loader = DataLoader([data], batch_size=1, collate_fn=pyg_collate_fn)
for i, b in enumerate(loader):
    print("Type of batch:", type(b))
    print("Keys:", b.keys)


Type of batch: <class 'abc.DataBatch'>
Keys: <bound method BaseData.keys of DataBatch(x=[3, 2], edge_index=[2, 2], batch=[3], ptr=[2])>


In [ ]:
if __name__ == "__main__":
    # Directories and parameter files
    data_dirs = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/{i}_matrix_output"
        for i in range(1, 11)
    ]
    param_files = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/{i}.py"
        for i in range(1, 11)
    ]

    # Run the main training process
    trained_model = main(data_dirs, param_files)

    # Save the final trained model
    torch.save(trained_model.state_dict(), "final_trained_model.pth")
    print("Final model saved to 'final_trained_model.pth'")

In [ ]:
import os

# Check if the files exist
directory = "/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/1_matrix_output"
timepoint = 0  # Replace with the correct timepoint
required_files = [
    f"{timepoint}_graph_mat.npy",
    f"{timepoint}_properties_mat.npy",
    f"{timepoint}_state_mat.npy"
]

for file in required_files:
    file_path = os.path.join(directory, file)
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
    else:
        print(f"File exists: {file_path}")